In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Dataset (Positive = 1, Negative = 0)
reviews = [
    ("Absolutely love this product! Exceeded all my expectations.", 1),
    ("Terrible quality, broke after two days of normal use.", 0),
    ("Great value for the money, fast shipping and works smoothly.", 1),
    ("Worst purchase ever. Customer support was completely unhelpful.", 0),
    ("Battery life is phenomenal and the screen is super crisp.", 1),
    ("Do not buy this. It overheats and shuts down randomly.", 0),
    ("Very satisfied with the build quality and sleek design.", 1),
    ("Arrived damaged and missing parts. Total waste of money.", 0),
    ("Works like a charm right out of the box. Highly recommend!", 1),
    ("Disappointing performance, lags constantly and feels cheap.", 0),
    ("Fantastic features for the price point, would buy again.", 1),
    ("Stopped charging within a week. Very frustrated.", 0)
]
df = pd.DataFrame(reviews, columns=['text', 'sentiment'])

# 2. Fundamental NLP Text Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation & numbers
    return re.sub(r'\s+', ' ', text).strip()

df['cleaned_text'] = df['text'].apply(clean_text)

# 3. Feature Extraction (TF-IDF with unigrams + bigrams) & Validation Split
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

val_clf = MultinomialNB(alpha=0.5)
val_clf.fit(X_train, y_train)
y_pred = val_clf.predict(X_test)

print("--- Model Validation Report (Holdout Set) ---")
print(f"Validation Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive'], zero_division=0))

# 4. Fit Production Model on Full Corpus for Live User Inference
prod_clf = MultinomialNB(alpha=0.5)
prod_clf.fit(X, y)
feature_names = np.array(vectorizer.get_feature_names_out())

def analyze_custom_review(raw_text):
    cleaned = clean_text(raw_text)
    vec = vectorizer.transform([cleaned])
    probs = prod_clf.predict_proba(vec)[0]  # [Neg_Prob, Pos_Prob]
    neg_pct, pos_pct = probs[0] * 100, probs[1] * 100

    # Identify recognized vocabulary tokens from the user's text
    nonzero_idx = vec.nonzero()[1]
    matched_terms = feature_names[nonzero_idx].tolist()

    if pos_pct >= 60:
        sentiment = "POSITIVE"
        action = "Route to public testimonials / request app store rating."
    elif neg_pct >= 60:
        sentiment = "NEGATIVE"
        action = "Flag for Customer Support escalation & refund/replacement check."
    else:
        sentiment = "NEUTRAL / UNCERTAIN (Out-of-Vocabulary Terms)"
        action = "Queue for manual review (contains words not yet in training corpus)."

    print("\n" + "=" * 60)
    print("REAL-TIME SENTIMENT ANALYSIS REPORT")
    print("=" * 60)
    print(f"Input Review      : \"{raw_text}\"")
    print(f"Predicted Sentiment: {sentiment}")
    print(f"Confidence Split  : Positive: {pos_pct:.1f}%  |  Negative: {neg_pct:.1f}%")
    print(f"Matched Keywords  : {', '.join(matched_terms) if matched_terms else 'None (unseen vocabulary)'}")
    print(f"Business Action   : {action}")
    print("=" * 60 + "\n")

# 5. Interactive User Input Loop (Type 'exit' to stop)
while True:
    user_review = input("Enter a product review to analyze (or type 'exit' to quit): ")
    if user_review.strip().lower() == 'exit':
        print("Exiting sentiment analyzer.")
        break
    if user_review.strip():
        analyze_custom_review(user_review)

--- Model Validation Report (Holdout Set) ---
Validation Accuracy: 50.00%
              precision    recall  f1-score   support

    Negative       0.50      0.50      0.50         2
    Positive       0.50      0.50      0.50         2

    accuracy                           0.50         4
   macro avg       0.50      0.50      0.50         4
weighted avg       0.50      0.50      0.50         4

Enter a product review to analyze (or type 'exit' to quit): exit
Exiting sentiment analyzer.


In [2]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

# 1. Dataset (Positive = 1, Negative = 0)
reviews = [
    ("Absolutely love this product! Exceeded all my expectations.", 1),
    ("Terrible quality, broke after two days of normal use.", 0),
    ("Great value for the money, fast shipping and works smoothly.", 1),
    ("Worst purchase ever. Customer support was completely unhelpful.", 0),
    ("Battery life is phenomenal and the screen is super crisp.", 1),
    ("Do not buy this. It overheats and shuts down randomly.", 0),
    ("Very satisfied with the build quality and sleek design.", 1),
    ("Arrived damaged and missing parts. Total waste of money.", 0),
    ("Works like a charm right out of the box. Highly recommend!", 1),
    ("Disappointing performance, lags constantly and feels cheap.", 0),
    ("Fantastic features for the price point, would buy again.", 1),
    ("Stopped charging within a week. Very frustrated.", 0)
]
df = pd.DataFrame(reviews, columns=['text', 'sentiment'])

# 2. Fundamental NLP Text Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation & numbers
    return text.strip()

df['cleaned_text'] = df['text'].apply(clean_text)

# 3. Feature Extraction (TF-IDF with built-in English stopwords removal)
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['sentiment']

# 4. Train & Evaluate Classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
clf = MultinomialNB()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("--- Classification Report ---")
print(classification_report(y_test, y_pred, zero_division=0))

# 5. Analyze Prediction Results on Unseen Text
sample_texts = ["Amazing build quality and fast performance", "Cheap plastic and broke immediately"]
sample_vecs = vectorizer.transform([clean_text(t) for t in sample_texts])
predictions = clf.predict(sample_vecs)

for text, pred in zip(sample_texts, predictions):
    label = "Positive" if pred == 1 else "Negative"
    print(f"Review: '{text}' -> Predicted Sentiment: {label}")

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.25      1.00      0.40         1
           1       0.00      0.00      0.00         3

    accuracy                           0.25         4
   macro avg       0.12      0.50      0.20         4
weighted avg       0.06      0.25      0.10         4

Review: 'Amazing build quality and fast performance' -> Predicted Sentiment: Negative
Review: 'Cheap plastic and broke immediately' -> Predicted Sentiment: Negative
